# 04i — Post-LoRA Attribution (L3 — Triangle Inequality Numeric (overfit))

**Runs on RunPod GPU. Always start from a FRESH kernel.**

Loads the merged LoRA checkpoint (`my_work/checkpoints/lora_l3_tri_num_merged/`)
produced by `04h_lora_l3_triangle_numeric_training.ipynb` and runs attribution on **every row** of the
frozen probe set (`prompts_triangle_v2.jsonl`) — **binary and numeric** `task_type`
slices, mirroring attribution targets from **`02`**.

**Why a separate notebook?**  Training leaves ~12 GB of GPU memory that cannot
be freed without a kernel restart (`empty_cache()` only releases the PyTorch
cache, not memory held by the CUDA process).  Starting fresh gives the full
19–24 GB to Gemma-2-2B + transcoders + attribution.

**Prerequisites:**
- `04_lora_training_data.ipynb` run (probe JSONL exists)
- `04h_lora_l3_triangle_numeric_training.ipynb` run through §7 (`lora_l3_tri_num_merged/` exists)

**Outputs:**
- Pilot: `my_work/results/statistics/stats_lora_l3_tri_num_pilot.json`
- Full:  `my_work/results/statistics/stats_lora_l3_tri_num.json`

**Checkpointing:** Stats append after each prompt. Re-running §5 skips **`prompt_id`s** already succeeded in **`STATS_FILE`**.

**Pilot to full:** **`RUN_FULL=True`** makes §5 copy successful pilot rows into the full stats file (instant, no GPU) so attribution is never paid twice; use **`stats_lora_l3_tri_num.json`** in **`05`** with **`USE_PILOT=False`** when the sweep is finished.

## 0 — Environment setup

In [1]:
import os
import sys
from pathlib import Path

# Must be set BEFORE torch is imported anywhere.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IN_RUNPOD = os.environ.get("RUNPOD_POD_ID") is not None

def _find_repo_root():
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "circuit_tracer" / "__init__.py").is_file():
            return directory
    workspace = Path("/workspace")
    if workspace.is_dir():
        for child in workspace.iterdir():
            if child.is_dir() and (child / "circuit_tracer" / "__init__.py").is_file():
                return child
    repo_override = os.environ.get("CT_REPO_DIR")
    if repo_override:
        p = Path(repo_override).expanduser().resolve()
        if (p / "circuit_tracer" / "__init__.py").is_file():
            return p
    return None

_root = _find_repo_root()
if _root is not None:
    if str(_root) not in sys.path:
        sys.path.insert(0, str(_root))
    _my_work = _root / "my_work"
    if str(_my_work) not in sys.path:
        sys.path.insert(0, str(_my_work))
    print(f"Repo root: {_root}")
else:
    print("WARNING: could not locate circuit_tracer repo. Set CT_REPO_DIR.")

try:
    from dotenv import load_dotenv
    if _root is not None and (_root / ".env").is_file():
        load_dotenv(_root / ".env")
except ImportError:
    pass

MY_WORK = _my_work if _root else Path(".").resolve()

if IN_RUNPOD:
    persistent_root = Path(os.environ.get("RUNPOD_PERSISTENT_ROOT", "/workspace")).resolve()
    hf_home = persistent_root / "hf"
    for key, path in {
        "HF_HOME": hf_home,
        "HUGGINGFACE_HUB_CACHE": hf_home / "hub",
        "TRANSFORMERS_CACHE": hf_home / "transformers",
    }.items():
        path.mkdir(parents=True, exist_ok=True)
        os.environ[key] = str(path)

STATS_DIR  = MY_WORK / "results" / "statistics"
GRAPHS_DIR = MY_WORK / "results" / "graphs_lora_l3_tri_num"
STATS_DIR.mkdir(parents=True, exist_ok=True)
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

print(f"MY_WORK  : {MY_WORK}")

Repo root: /workspace/thesis_circuit_breaker
MY_WORK  : /workspace/thesis_circuit_breaker/my_work


## 1 — Constants

In [2]:
import json
import time
import gc
import traceback
import random

import torch

MODEL_NAME      = "google/gemma-2-2b"
TRANSCODER_NAME = "gemma"
TOKEN_TRUE      = " True"
TOKEN_FALSE     = " False"
VOCAB_ID_TRUE   = 5569
VOCAB_ID_FALSE  = 7662

ATTR_KWARGS = dict(
    batch_size=256,
    max_feature_nodes=8192,
    offload="disk",
    verbose=False,
)
PHASE = "lora_l3_tri_num"

# Pilot vs full
# Start RUN_FULL=False for ~N_PILOT prompts. Set RUN_FULL=True to run everyone;
# §5 skips rows already saved and copies pilot successes into the full file (no rerun).
RUN_FULL = True
N_PILOT  = 40

MERGED_DIR       = MY_WORK / "checkpoints" / "lora_l3_tri_num_merged"
PROBE_JSONL      = MY_WORK / "data" / "prompts_triangle_v2.jsonl"
PILOT_STATS_FILE = STATS_DIR / "stats_lora_l3_tri_num_pilot.json"
FULL_STATS_FILE  = STATS_DIR / "stats_lora_l3_tri_num.json"
STATS_FILE       = FULL_STATS_FILE if RUN_FULL else PILOT_STATS_FILE

assert MERGED_DIR.exists(), (
    f"Merged checkpoint not found: {MERGED_DIR}\n"
    "Run 04h_lora_l3_triangle_numeric_training.ipynb through §7 first."
)

device_str = "cuda" if torch.cuda.is_available() else "cpu"
device     = torch.device(device_str)

print(f"MERGED_DIR : {MERGED_DIR}")
print(f"STATS_FILE : {STATS_FILE}")
print(f"RUN_FULL   : {RUN_FULL}  (pilot N={N_PILOT})")
print(f"CUDA       : {torch.cuda.is_available()}", end="")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"  |  Free VRAM: {free/1e9:.1f} GB / {total/1e9:.1f} GB")
else:
    print()

MERGED_DIR : /workspace/thesis_circuit_breaker/my_work/checkpoints/lora_l3_tri_num_merged
STATS_FILE : /workspace/thesis_circuit_breaker/my_work/results/statistics/stats_lora_l3_tri_num.json
RUN_FULL   : True  (pilot N=40)
CUDA       : True  |  Free VRAM: 20.8 GB / 21.0 GB


## 2 — Load merged model into ReplacementModel

TransformerLens requires a registered `model_name`; pass the local merged
weights via `hf_model=`.  We load `merged_hf` on CPU so that only one copy
of Gemma-2-2B lives on the GPU (inside HookedTransformer) at any time.

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from circuit_tracer import ReplacementModel

print("Loading merged weights on CPU ...")
merged_hf = AutoModelForCausalLM.from_pretrained(
    str(MERGED_DIR),
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map="cpu",
).eval()

print("Building ReplacementModel (moves weights to GPU) ...")
ct_model = ReplacementModel.from_pretrained(
    MODEL_NAME,
    TRANSCODER_NAME,
    dtype=torch.bfloat16,
    backend="transformerlens",
    device=device,
    lazy_encoder=True,
    lazy_decoder=True,
    hf_model=merged_hf,
)

# Free the CPU copy immediately — HookedTransformer now holds the weights on GPU.
del merged_hf
gc.collect()

ct_tokenizer = ct_model.tokenizer

id_true  = ct_tokenizer.encode(TOKEN_TRUE,  add_special_tokens=False)[-1]
id_false = ct_tokenizer.encode(TOKEN_FALSE, add_special_tokens=False)[-1]
assert id_true  == VOCAB_ID_TRUE,  f"Token mismatch True:  {id_true}"
assert id_false == VOCAB_ID_FALSE, f"Token mismatch False: {id_false}"
print(f"ReplacementModel ready. Token IDs: True={id_true}, False={id_false}")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM after load: {(total-free)/1e9:.1f} GB used / {total/1e9:.1f} GB total")

/workspace/venvs/ct/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading merged weights on CPU ...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Building ReplacementModel (moves weights to GPU) ...


Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer
ReplacementModel ready. Token IDs: True=5569, False=7662
VRAM after load: 8.6 GB used / 21.0 GB total


## 3 — Load full probe set (binary + numeric) and optional pilot subset

In [4]:
from collections import Counter, defaultdict

def _resolve_attr_targets(entry: dict) -> list[str]:
    """Same logic as `02_baseline_attribution.ipynb` (single-token numeric targets)."""
    task_type = entry.get("task_type", "binary")
    if task_type == "binary":
        return [TOKEN_TRUE, TOKEN_FALSE]
    raw_label = entry["label_token"]
    ids_raw = ct_tokenizer.encode(raw_label, add_special_tokens=False)
    if len(ids_raw) == 1:
        return [raw_label]
    alt = raw_label.lstrip()
    ids_alt = ct_tokenizer.encode(alt, add_special_tokens=False)
    if len(ids_alt) == 1:
        return [alt]
    raise ValueError(
        f"Numeric attribution target must be single-token, "
        f"got {raw_label!r}->{ids_raw}, alt {alt!r}->{ids_alt}"
    )

all_probe = []
with open(PROBE_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        all_probe.append(json.loads(line))

print(f"All probe prompts: {len(all_probe)}")
print("  by task_type:", dict(Counter(p.get("task_type", "binary") for p in all_probe)))

if RUN_FULL:
    probe_prompts = all_probe
    print(f"RUN_FULL=True: using all {len(probe_prompts)} prompts.")
else:
    strata: dict = defaultdict(list)
    for p in all_probe:
        key = (p.get("family", "?"), p.get("tail", "?"), str(p.get("label", "?")))
        strata[key].append(p)

    rng = random.Random(42)
    pilot = []
    per_stratum = max(1, N_PILOT // len(strata))
    for key, bucket in sorted(strata.items()):
        rng.shuffle(bucket)
        pilot.extend(bucket[:per_stratum])

    remaining_pool = [p for p in all_probe if p not in pilot]
    rng.shuffle(remaining_pool)
    while len(pilot) < N_PILOT and remaining_pool:
        pilot.append(remaining_pool.pop(0))

    probe_prompts = pilot[:N_PILOT]
    dist = Counter(
        (p.get("family", "?"), p.get("tail", "?"), str(p.get("label", "?")))
        for p in probe_prompts
    )
    print(f"Pilot subset: {len(probe_prompts)} prompts")
    for key, cnt in sorted(dist.items()):
        print(f"  {key}: {cnt}")

All probe prompts: 300
  by task_type: {'binary': 270, 'numeric': 30}
RUN_FULL=True: using all 300 prompts.


## 4 — First-token accuracy (post-LoRA), binary + numeric

In [5]:
overall_correct = 0

binary_correct = 0
binary_total = 0
correct_true = 0
total_true = 0
correct_false = 0
total_false = 0

numeric_correct = 0
numeric_total = 0

predictions = []

with torch.inference_mode():
    for entry in probe_prompts:
        logits, _ = ct_model.feature_intervention(entry["prompt"], [])
        last_logit = logits.squeeze(0)[-1, :]
        pred_id = int(last_logit.argmax().item())
        pred_tok = ct_tokenizer.decode([pred_id])

        task_type = entry.get("task_type", "binary")
        if task_type == "binary":
            label_id = VOCAB_ID_TRUE if entry["label"] else VOCAB_ID_FALSE
            binary_total += 1
            is_ok = pred_id == label_id
            if is_ok:
                binary_correct += 1
                overall_correct += 1
            if entry["label"]:
                total_true += 1
                if pred_id == VOCAB_ID_TRUE:
                    correct_true += 1
            else:
                total_false += 1
                if pred_id == VOCAB_ID_FALSE:
                    correct_false += 1
        else:
            raw_label = entry["label_token"]
            label_ids = ct_tokenizer.encode(raw_label, add_special_tokens=False)
            if len(label_ids) != 1:
                alt = raw_label.lstrip()
                alt_ids = ct_tokenizer.encode(alt, add_special_tokens=False)
                if len(alt_ids) == 1:
                    label_ids = alt_ids
                else:
                    raise ValueError(
                        f"Numeric label_token must encode to single id, "
                        f"got {raw_label!r}->{label_ids}, alt {alt!r}->{alt_ids}"
                    )
            label_id = int(label_ids[0])
            numeric_total += 1
            is_ok = pred_id == label_id
            if is_ok:
                numeric_correct += 1
                overall_correct += 1

        predictions.append({
            "prompt_id": entry["prompt_id"],
            "task_type": task_type,
            "pred_token": pred_tok,
            "pred_id": pred_id,
            "is_correct": is_ok,
        })

n = len(probe_prompts)
print(f"Post-LoRA first-token accuracy (n={n}):")
print(f"  Overall : {overall_correct}/{n} = {overall_correct/n:.1%}")
if binary_total:
    print(f"  Binary  : {binary_correct}/{binary_total} = {binary_correct/binary_total:.1%}")
    if total_true:
        print(f"    True : {correct_true}/{total_true} = {correct_true/total_true:.1%}")
    if total_false:
        print(f"    False: {correct_false}/{total_false} = {correct_false/total_false:.1%}")
if numeric_total:
    print(f"  Numeric : {numeric_correct}/{numeric_total} = {numeric_correct/numeric_total:.1%}")
print("(Baseline notebook `02` uses the same split for comparison.)")

Post-LoRA first-token accuracy (n=300):
  Overall : 30/300 = 10.0%
  Binary  : 0/270 = 0.0%
    True : 0/135 = 0.0%
    False: 0/135 = 0.0%
  Numeric : 30/30 = 100.0%
(Baseline notebook `02` uses the same split for comparison.)


In [6]:
import pandas as pd

_csv_path = MY_WORK / "results" / "first_token" / "lora_l3_tri_num_predictions.csv"
_csv_path.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(predictions).to_csv(_csv_path, index=False)
print(f"First-token rows saved → {_csv_path}")

First-token rows saved → /workspace/thesis_circuit_breaker/my_work/results/first_token/lora_l3_tri_num_predictions.csv


## 4b — First-token probability tables (monitoring)

Runs **two passes** over **`probe_prompts`**: (**1**) binary rows → softmax probability on **`VOCAB_ID_TRUE`** / **`VOCAB_ID_FALSE`** at the **last prompt position**; (**2**) numeric rows → softmax mass on the **single token id** Gemma assigns to each digit string **`"0"` … `"9"`** (tries plain digit then a leading space, same spirit as numeric targets in §3).

Results: **`df_binary_token_probs`** and **`df_numeric_digit_probs`**. Progress prints every **`PROB_MONITOR_EVERY`** rows.

In [7]:
import pandas as pd
import torch.nn.functional as F
from IPython.display import display

PROB_MONITOR_EVERY = max(5, max(1, len(probe_prompts)) // 20)


def _last_vocab_probs(prompt: str) -> torch.Tensor:
    logits, _ = ct_model.feature_intervention(prompt, [])
    last = logits.squeeze(0)[-1, :].float()
    return F.softmax(last, dim=-1)


def _resolve_digit_vocab_id(d: int) -> int:
    """Single-token vocab id Gemma assigns to digit continuation (plain or spaced)."""
    ch = str(int(d))
    for variant in (ch, f" {ch}"):
        tids = ct_tokenizer.encode(variant, add_special_tokens=False)
        if len(tids) == 1:
            return int(tids[0])
    raise ValueError(f"Digit {d!r} must map to a single token; tried {ch!r} and {(' '+ch)!r}")


DIGIT_IDS = [_resolve_digit_vocab_id(d) for d in range(10)]
print(f"Digit vocab ids [0..9]: {DIGIT_IDS}")

binary_list = [p for p in probe_prompts if p.get("task_type", "binary") == "binary"]
numeric_list = [p for p in probe_prompts if p.get("task_type", "binary") == "numeric"]

# --- Pass 1: binary ---
t0 = time.time()
binary_rows = []
for i, entry in enumerate(binary_list, 1):
    probs = _last_vocab_probs(entry["prompt"])
    pt = float(probs[VOCAB_ID_TRUE].item())
    pf = float(probs[VOCAB_ID_FALSE].item())
    binary_rows.append(
        {
            "prompt_id": entry["prompt_id"],
            "family": entry.get("family"),
            "tail": entry.get("tail"),
            "label": entry.get("label"),
            "task_type": entry.get("task_type", "binary"),
            "prob_true": pt,
            "prob_false": pf,
            "prob_mass_tf": pt + pf,
        }
    )
    if PROB_MONITOR_EVERY and (i % PROB_MONITOR_EVERY == 0 or i == len(binary_list)):
        dt = max(time.time() - t0, 1e-6)
        print(
            f"  binary {i}/{len(binary_list)}  "
            f"{i/dt:.2f}/s  ETA {(len(binary_list)-i)/max(i/dt,1e-6):.1f}s"
        )

t1 = time.time()
df_binary_token_probs = pd.DataFrame(binary_rows)
print(f"df_binary_token_probs rows={len(df_binary_token_probs)}  wall={t1-t0:.1f}s")

# --- Pass 2: numeric digit masses ---
numeric_rows = []
t0 = time.time()
for i, entry in enumerate(numeric_list, 1):
    probs = _last_vocab_probs(entry["prompt"])
    row = {
        "prompt_id": entry["prompt_id"],
        "family": entry.get("family"),
        "tail": entry.get("tail"),
        "label": entry.get("label"),
        "label_token": entry.get("label_token"),
        "task_type": entry.get("task_type"),
    }
    mass = 0.0
    for d in range(10):
        p = float(probs[DIGIT_IDS[d]].item())
        row[f"prob_digit_{d}"] = p
        mass += p
    row["prob_mass_digits"] = mass
    numeric_rows.append(row)
    if PROB_MONITOR_EVERY and (i % PROB_MONITOR_EVERY == 0 or i == len(numeric_list)):
        dt = max(time.time() - t0, 1e-6)
        print(
            f"  numeric {i}/{len(numeric_list)}  "
            f"{i/dt:.2f}/s  ETA {(len(numeric_list)-i)/max(i/dt,1e-6):.1f}s"
        )

t1 = time.time()
df_numeric_digit_probs = pd.DataFrame(numeric_rows)
print(f"df_numeric_digit_probs rows={len(df_numeric_digit_probs)}  wall={t1-t0:.1f}s")

display(df_binary_token_probs.head(3))
display(df_numeric_digit_probs.head(3))

FIRST_TOKEN_DIR = MY_WORK / "results" / "first_token"
FIRST_TOKEN_DIR.mkdir(parents=True, exist_ok=True)
stem = PHASE + ("_pilot" if not RUN_FULL else "")
path_bin = FIRST_TOKEN_DIR / f"lora_binary_token_probs_{stem}.csv"
path_num = FIRST_TOKEN_DIR / f"lora_numeric_digit_probs_{stem}.csv"
df_binary_token_probs.to_csv(path_bin, index=False)
df_numeric_digit_probs.to_csv(path_num, index=False)
print(f"Wrote {path_bin}")
print(f"Wrote {path_num}")

Digit vocab ids [0..9]: [235276, 235274, 235284, 235304, 235310, 235308, 235318, 235324, 235321, 235315]
  binary 15/270  0.79/s  ETA 322.5s
  binary 30/270  0.79/s  ETA 302.4s
  binary 45/270  0.79/s  ETA 283.2s
  binary 60/270  0.80/s  ETA 263.8s
  binary 75/270  0.79/s  ETA 246.0s
  binary 90/270  0.79/s  ETA 228.3s
  binary 105/270  0.79/s  ETA 209.5s
  binary 120/270  0.79/s  ETA 190.8s
  binary 135/270  0.78/s  ETA 172.0s
  binary 150/270  0.79/s  ETA 152.8s
  binary 165/270  0.78/s  ETA 133.8s
  binary 180/270  0.78/s  ETA 114.7s
  binary 195/270  0.78/s  ETA 95.6s
  binary 210/270  0.78/s  ETA 76.6s
  binary 225/270  0.78/s  ETA 57.4s
  binary 240/270  0.79/s  ETA 38.2s
  binary 255/270  0.79/s  ETA 19.1s
  binary 270/270  0.79/s  ETA 0.0s
df_binary_token_probs rows=270  wall=343.2s
  numeric 15/30  0.79/s  ETA 19.0s
  numeric 30/30  0.79/s  ETA 0.0s
df_numeric_digit_probs rows=30  wall=38.0s


,prompt_id,family,tail,label,task_type,prob_true,prob_false,prob_mass_tf
0,tri_v2_001,numeric_validity,answer_colon,False,binary,3.068242e-13,1.658676e-12,1.965500e-12
1,tri_v2_002,numeric_validity,answer_colon,True,binary,3.578342e-10,2.435092e-11,3.821851e-10
2,tri_v2_003,numeric_validity,true_or_false,False,binary,1.588395e-04,2.938236e-05,1.882219e-04


,prompt_id,family,tail,label,label_token,task_type,prob_digit_0,prob_digit_1,prob_digit_2,prob_digit_3,prob_digit_4,prob_digit_5,prob_digit_6,prob_digit_7,prob_digit_8,prob_digit_9,prob_mass_digits
0,tri_v2_009,numeric_open,answer_colon,6,6,numeric,1.728382e-11,1.277111e-10,1.199735e-10,1.359478e-10,2.276697e-10,2.017644e-08,9.999999e-01,5.634880e-10,6.188705e-10,4.058651e-10,1.000000
1,tri_v2_010,numeric_open,answer_colon,2,2,numeric,2.646564e-09,2.896215e-07,9.999964e-01,5.804985e-07,7.237465e-08,4.205117e-08,4.511432e-08,5.570875e-08,1.325984e-07,2.224946e-07,0.999998
2,tri_v2_024,numeric_open,answer_colon,3,3,numeric,1.573710e-11,1.192904e-09,3.345582e-09,9.999999e-01,4.644881e-09,4.397685e-09,1.484593e-09,2.093612e-09,3.293714e-09,1.022492e-08,1.000000


Wrote /workspace/thesis_circuit_breaker/my_work/results/first_token/lora_binary_token_probs_lora_l3_tri_num.csv
Wrote /workspace/thesis_circuit_breaker/my_work/results/first_token/lora_numeric_digit_probs_lora_l3_tri_num.csv


## 5 — Attribution + stats loop

Same **`compute_statistics`** schema as **`02`**. Compare in `05_lora_comparison.ipynb` on **`prompt_id`**.

**Checkpoint / resume**

- Any row with **`attribution_succeeded`** already in **`STATS_FILE`** is skipped (no rerun).

- **`RUN_FULL=False`** only appends to **`stats_lora_l3_tri_num_pilot.json`**.

- **`RUN_FULL=True`** appends to **`stats_lora_l3_tri_num.json`**. Prompts that succeeded in **`stats_lora_l3_tri_num_pilot.json`** but are still missing from the full file have their rows **copied** over (instant, no GPU) so you avoid duplicate attribution **and** a completed full run fills **`stats_lora_l3_tri_num.json`** for **`USE_PILOT=False`** in `05`.

In [ ]:
import importlib
import utils.graph_statistics as gs_mod
importlib.reload(gs_mod)

from circuit_tracer import attribute
from utils.graph_statistics import compute_statistics, load_statistics, append_statistic

def _successful_rows_by_id(rows):
    return {
        e["prompt_id"]: e
        for e in rows
        if e.get("prompt_id") is not None and e.get("attribution_succeeded")
    }

existing = load_statistics(STATS_FILE)
done_rows = _successful_rows_by_id(existing)
done_ids = set(done_rows)

pilot_succ: dict = {}
if RUN_FULL and STATS_FILE.resolve() != PILOT_STATS_FILE.resolve():
    pilot_succ = _successful_rows_by_id(load_statistics(PILOT_STATS_FILE))

n_prev_failed = sum(1 for e in existing if not e.get("attribution_succeeded"))
remaining = [p for p in probe_prompts if p["prompt_id"] not in done_ids]
n_copy_hint = sum(1 for p in remaining if p["prompt_id"] in pilot_succ)

print(f"Stats file (append target) : {STATS_FILE}")
print(f"Already succeeded (here) : {len(done_ids)}")
if RUN_FULL and pilot_succ:
    print(f"Pilot rows to reuse      : {n_copy_hint}")
print(f"Previously failed (here) : {n_prev_failed} (eligible for retry)")
print(f"Remaining to process     : {len(remaining)} prompts")
print()

t0 = time.time()
n_success = 0
n_fail = 0
n_copied = 0
progress_every = 5

for i, entry in enumerate(remaining, start=1):
    pid = entry["prompt_id"]
    task_type = entry.get("task_type", "binary")
    print(f"[{i}/{len(remaining)}] {pid} ({task_type}) ...", end=" ", flush=True)

    if RUN_FULL and pid in pilot_succ:
        append_statistic(dict(pilot_succ[pid]), STATS_FILE)
        n_success += 1
        n_copied += 1
        print("OK (reused pilot row)")
        continue

    try:
        graph = attribute(
            prompt=entry["prompt"],
            model=ct_model,
            attribution_targets=_resolve_attr_targets(entry),
            **ATTR_KWARGS,
        )
        stat = compute_statistics(graph, entry, phase=PHASE)
        append_statistic(stat, STATS_FILE)
        n_success += 1

        del graph
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        density  = stat.get("edge_density", float("nan"))
        n_active = stat.get("n_active_features", "?")
        print(f"OK  n_active={n_active}  density={density:.4f}")

    except Exception as exc:
        n_fail += 1
        stat = {
            "prompt_id": pid,
            "phase": PHASE,
            "task_type": task_type,
            "family": entry.get("family"),
            "tail": entry.get("tail"),
            "label": entry["label"],
            "label_token": entry["label_token"],
            "template_id": entry.get("template_id"),
            "attribution_succeeded": False,
            "_error": str(exc),
        }
        append_statistic(stat, STATS_FILE)
        print(f"FAIL: {exc}")
        traceback.print_exc()

    if i % progress_every == 0:
        elapsed = time.time() - t0
        speed   = i / elapsed
        eta     = (len(remaining) - i) / speed if speed > 0 else float("inf")
        print(
            f"[progress] {i}/{len(remaining)} ({i/len(remaining):.0%}) | "
            f"success={n_success}/{n_success+n_fail} | "
            f"{speed:.3f} prompts/s | ETA {eta/60:.1f} min"
        )

print()
print(f"Done. {n_success} ok ({n_copied} reused from pilot), {n_fail} failed in {time.time()-t0:.0f}s.")
print(f"Stats saved to: {STATS_FILE}")

Stats file (append target) : /workspace/thesis_circuit_breaker/my_work/results/statistics/stats_lora_l3_tri_num.json
Already succeeded (here) : 0
Previously failed (here) : 0 (eligible for retry)
Remaining to process     : 300 prompts

[1/300] tri_v2_001 (binary) ... OK  n_active=19601  density=0.1325
[2/300] tri_v2_002 (binary) ... OK  n_active=16364  density=0.1394
[3/300] tri_v2_003 (binary) ... OK  n_active=28913  density=0.1171
[4/300] tri_v2_004 (binary) ... OK  n_active=24400  density=0.1189
[5/300] tri_v2_005 (binary) ... OK  n_active=25620  density=0.1179
[progress] 5/300 (2%) | success=5/5 | 0.013 prompts/s | ETA 388.3 min
[6/300] tri_v2_006 (binary) ... OK  n_active=18786  density=0.1357
[7/300] tri_v2_007 (binary) ... OK  n_active=27221  density=0.1197
[8/300] tri_v2_008 (binary) ... OK  n_active=16355  density=0.1235
[9/300] tri_v2_009 (numeric) ... OK  n_active=28468  density=0.1436
[10/300] tri_v2_010 (numeric) ... OK  n_active=28014  density=0.1355
[progress] 10/300 (3%

## 6 — Scale-up decision

After the pilot: if **`n_success >= N_PILOT * 0.9`** (or whenever you want the full probe set),

set **`RUN_FULL = True`** in §1 and **re-run from §1**.

§5 skips IDs already recorded in **`stats_lora_l3_tri_num.json`** (if any) **and**

**reuses pilot successes by copying rows** — no duplicate attribution cost for prompts that finished during the pilot.

In [ ]:
final_stats = load_statistics(STATS_FILE)
n_ok    = sum(1 for s in final_stats if s.get("attribution_succeeded"))
n_total = len(final_stats)

print(f"=== Post-run summary ===")
print(f"Stats file : {STATS_FILE}")
print(f"Total rows : {n_total}")
print(f"Succeeded  : {n_ok}  ({n_ok/max(n_total,1):.1%})")
print()
if not RUN_FULL:
    print("Pilot complete.")
    print("If results look good: set RUN_FULL=True in §1 and re-run from §1 onwards.")
    print("Full run saves to:", FULL_STATS_FILE)
else:
    print("Full run complete. Proceed to 05_lora_comparison.ipynb.")